## Chunked Recording Integration Test

End-to-end validation of the chunked recording server-side features:

1. Pair device, create project + capture session
2. Upload 3 audio chunks (chunk_index 0, 1, 2) with `is_final_chunk=true` on chunk 2
3. Verify all 3 stored with correct `chunk_index` values
4. Upload duplicate chunk 1 → verify dedup (same row returned, `_dedup: true`)
5. Call `GET /api/captures/:id/audio/chunks` → verify ordered response
6. Call `GET /api/captures/:id/audio/stream` → verify valid M4A stream
7. Verify OTel logs for chunk events

Run cells top-to-bottom. Parameterized for post-deploy job.

In [0]:
%pip install --upgrade databricks-sdk cryptography
dbutils.library.restartPython()

In [0]:
import sys
import os

# Derive tests directory from this notebook's location (portable across deploy targets)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
tests_dir = '/Workspace' + os.path.dirname(notebook_path)
sys.path.insert(0, tests_dir)

from databricks.sdk import WorkspaceClient
from lib.pairing_client import PairingTestClient

# ── Job parameters (widgets) ────────────────────────────────────────────────────
dbutils.widgets.text('app_name', 'lakeloom-ai-dev', 'App Name')
dbutils.widgets.text('catalog_use', 'hls_fde_dev', 'Catalog')
dbutils.widgets.text('schema_use', 'dev_matthew_giglia_lakeloom', 'Schema')

APP_NAME = dbutils.widgets.get('app_name')
CATALOG = dbutils.widgets.get('catalog_use')
SCHEMA = dbutils.widgets.get('schema_use')

wc = WorkspaceClient()
WORKSPACE_HOST = wc.config.host.rstrip('/')

SCOPE = 'lakeloom_credentials'
XCODE_CLIENT_ID = dbutils.secrets.get(SCOPE, 'xcode_client_id_dev_matthew_giglia_lakeloom')
XCODE_CLIENT_SECRET = dbutils.secrets.get(SCOPE, 'xcode_client_secret_dev_matthew_giglia_lakeloom')

APP_URL = None
for app in wc.apps.list():
    if app.name == APP_NAME:
        APP_URL = app.url.rstrip('/')
        break

if not APP_URL:
    raise ValueError(f'Could not find app URL for {APP_NAME}')

print('Workspace host:', WORKSPACE_HOST)
print('App name:      ', APP_NAME)
print('Catalog:       ', CATALOG)
print('Schema:        ', SCHEMA)
print('App url:       ', APP_URL)
print('Tests dir:     ', tests_dir)

In [0]:
import uuid
import time
import json

# ── Pair device ─────────────────────────────────────────────────────────────────
client = PairingTestClient(
    app_url=APP_URL,
    workspace_host=WORKSPACE_HOST,
    xcode_client_id=XCODE_CLIENT_ID,
    xcode_client_secret=XCODE_CLIENT_SECRET,
)
client.acquire_spn_token()

TEST_DEVICE_ID = str(uuid.uuid4())
session = client.pair_device(
    device_label=f'Chunked Recording Test {int(time.time())}',
    device_id=TEST_DEVICE_ID,
)

SPN_TOKEN = session.spn_token
PAIRED_SESSION_ID = session.paired_session_id
print(f'Device paired: {PAIRED_SESSION_ID}')

# ── Create project ──────────────────────────────────────────────────────────────
project_body = {
    'name': f'Chunked Recording Test {int(time.time())}',
    'description': 'Temporary project for chunked recording verification',
    'workspace_id': WORKSPACE_HOST,
    'client_generated_id': str(uuid.uuid4()),
}
project_resp = session.post('/api/v1/projects', json_body=project_body)
project_resp.raise_for_status()
project_data = project_resp.json()


def extract_id(payload):
    if isinstance(payload, dict):
        for id_key in ('id', 'project_id'):
            val = payload.get(id_key)
            if isinstance(val, str) and val:
                return val
        for key in ('project', 'data', 'result', 'item'):
            nested = payload.get(key)
            if nested is not None:
                found = extract_id(nested)
                if found:
                    return found
    return None


PROJECT_ID = extract_id(project_data)
if not PROJECT_ID:
    raise RuntimeError(f'No project id in response: {json.dumps(project_data)[:500]}')
print(f'Project created: {PROJECT_ID}')

# ── Assign device ───────────────────────────────────────────────────────────────
assign_resp = session.post(
    f'/api/v1/projects/{PROJECT_ID}/devices',
    json_body={'paired_session_id': PAIRED_SESSION_ID},
)
assign_resp.raise_for_status()
print(f'Device assigned')

# ── Create capture session ──────────────────────────────────────────────────────
capture_body = {
    'label': f'Chunked Recording Capture {int(time.time())}',
    'device_id': TEST_DEVICE_ID,
}
capture_resp = session.post(f'/api/projects/{PROJECT_ID}/captures', json_body=capture_body)
capture_resp.raise_for_status()
capture_data = capture_resp.json()
CAPTURE_ID = extract_id(capture_data)
if not CAPTURE_ID:
    raise RuntimeError(f'No capture id in response: {json.dumps(capture_data)[:500]}')
print(f'Capture created: {CAPTURE_ID}')

In [0]:
import hashlib
import struct
import io
import wave


def build_test_wav(sample_rate=16000, duration_ms=500, tone_hz=440):
    """Build a short WAV file with a sine tone (distinguishable per chunk)."""
    import math
    num_frames = int(sample_rate * duration_ms / 1000)
    wav_buffer = io.BytesIO()
    with wave.open(wav_buffer, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        frames = b''.join(
            struct.pack('<h', int(16000 * math.sin(2 * math.pi * tone_hz * i / sample_rate)))
            for i in range(num_frames)
        )
        wf.writeframes(frames)
    return wav_buffer.getvalue()


# Build 3 distinct chunks (different tones so SHA-256 differs)
CHUNK_0_BYTES = build_test_wav(tone_hz=440)   # A4
CHUNK_1_BYTES = build_test_wav(tone_hz=523)   # C5
CHUNK_2_BYTES = build_test_wav(tone_hz=659)   # E5

CHUNK_0_SHA = hashlib.sha256(CHUNK_0_BYTES).hexdigest()
CHUNK_1_SHA = hashlib.sha256(CHUNK_1_BYTES).hexdigest()
CHUNK_2_SHA = hashlib.sha256(CHUNK_2_BYTES).hexdigest()

print(f'Chunk 0: {len(CHUNK_0_BYTES):,} bytes, SHA: {CHUNK_0_SHA[:16]}...')
print(f'Chunk 1: {len(CHUNK_1_BYTES):,} bytes, SHA: {CHUNK_1_SHA[:16]}...')
print(f'Chunk 2: {len(CHUNK_2_BYTES):,} bytes, SHA: {CHUNK_2_SHA[:16]}...')
print(f'All unique: {len({CHUNK_0_SHA, CHUNK_1_SHA, CHUNK_2_SHA}) == 3}')

In [0]:
upload_results = []

for idx, (chunk_bytes, chunk_sha) in enumerate([
    (CHUNK_0_BYTES, CHUNK_0_SHA),
    (CHUNK_1_BYTES, CHUNK_1_SHA),
    (CHUNK_2_BYTES, CHUNK_2_SHA),
]):
    is_final = idx == 2  # Last chunk
    client_ts = str(int(time.time()) + idx)  # Slightly different timestamps

    resp = session.upload(
        path=f'/api/captures/{CAPTURE_ID}/audio',
        file_bytes=chunk_bytes,
        filename=f'chunk-{idx}-recording.wav',
        mime_type='audio/wav',
        extra_fields={
            'client_ts': client_ts,
            'sha256_hex': chunk_sha,
            'device_id': TEST_DEVICE_ID,
            'chunk_index': str(idx),
            'is_final_chunk': 'true' if is_final else 'false',
            'total_chunks': '3',
        },
    )
    resp.raise_for_status()
    data = resp.json()
    upload_results.append(data)

    status = '\u2705' if data.get('chunk_index') == idx else '\u274c'
    final_flag = ' (FINAL)' if data.get('is_final_chunk') else ''
    print(f'{status} Chunk {idx}: upload_id={data["id"][:12]}..., '
          f'chunk_index={data.get("chunk_index")}, '
          f'is_final_chunk={data.get("is_final_chunk")}{final_flag}')

print(f'\n\u2705 All {len(upload_results)} chunks uploaded successfully')

In [0]:
print('=' * 60)
print('CHUNK INDEX VERIFICATION')
print('=' * 60)

assertions = []

for idx, result in enumerate(upload_results):
    ci = result.get('chunk_index')
    ifc = result.get('is_final_chunk')
    expected_final = (idx == 2)

    ok_idx = ci == idx
    ok_final = ifc == expected_final

    assertions.append(ok_idx and ok_final)
    status = '\u2705' if (ok_idx and ok_final) else '\u274c'
    print(f'{status} Chunk {idx}: chunk_index={ci} (expect {idx}), '
          f'is_final_chunk={ifc} (expect {expected_final})')

assert all(assertions), 'Some chunk assertions failed!'
print(f'\n\u2705 All chunk_index + is_final_chunk values correct')

In [0]:
print('=' * 60)
print('DEDUP TEST: Re-upload chunk 1 (same SHA — clean retry)')
print('=' * 60)

dedup_resp = session.upload(
    path=f'/api/captures/{CAPTURE_ID}/audio',
    file_bytes=CHUNK_1_BYTES,
    filename='chunk-1-recording.wav',
    mime_type='audio/wav',
    extra_fields={
        'client_ts': str(int(time.time())),
        'sha256_hex': CHUNK_1_SHA,
        'device_id': TEST_DEVICE_ID,
        'chunk_index': '1',
        'is_final_chunk': 'false',
    },
)

print(f'HTTP status: {dedup_resp.status_code}')
dedup_data = dedup_resp.json()

assert dedup_resp.status_code == 201, f'Expected 201, got {dedup_resp.status_code}'

original_id = upload_results[1]['id']
dedup_id = dedup_data.get('id')
assert dedup_id == original_id, f'Dedup returned different id: {dedup_id} != {original_id}'
print(f'\u2705 DEDUP VERIFIED -- same upload_id returned: {dedup_id[:12]}...')

assert dedup_data.get('_dedup') == True, f'Expected _dedup=True, got {dedup_data.get("_dedup")}'
print(f'\u2705 _dedup flag: {dedup_data["_dedup"]}')

assert dedup_data.get('dedup_sha_mismatch') == False, (
    f'Expected dedup_sha_mismatch=False for same-SHA retry, got {dedup_data.get("dedup_sha_mismatch")}'
)
print(f'\u2705 dedup_sha_mismatch: {dedup_data["dedup_sha_mismatch"]} (same SHA = clean retry)')

In [0]:
print('=' * 60)
print('DEDUP TEST: Different file at chunk_index=1 (SHA mismatch)')
print('=' * 60)

# Upload chunk 0's bytes but claim chunk_index=1 -- different SHA
different_sha = CHUNK_0_SHA  # This differs from CHUNK_1_SHA stored at index 1

mismatch_resp = session.upload(
    path=f'/api/captures/{CAPTURE_ID}/audio',
    file_bytes=CHUNK_0_BYTES,
    filename='chunk-1-recording-recovery.wav',
    mime_type='audio/wav',
    extra_fields={
        'client_ts': str(int(time.time())),
        'sha256_hex': different_sha,
        'device_id': TEST_DEVICE_ID,
        'chunk_index': '1',
        'is_final_chunk': 'false',
    },
)

print(f'HTTP status: {mismatch_resp.status_code}')
mismatch_data = mismatch_resp.json()

# Still returns 201 (idempotent, first file wins)
assert mismatch_resp.status_code == 201, f'Expected 201, got {mismatch_resp.status_code}'

# Returns the original chunk 1 upload id (first file wins)
assert mismatch_data.get('id') == original_id, (
    f'Expected original id {original_id[:12]}..., got {mismatch_data.get("id", "?")[:12]}...'
)
print(f'\u2705 First file wins -- original upload_id returned: {mismatch_data["id"][:12]}...')

# _dedup should be true
assert mismatch_data.get('_dedup') == True
print(f'\u2705 _dedup flag: {mismatch_data["_dedup"]}')

# dedup_sha_mismatch should be TRUE (different file at same slot)
assert mismatch_data.get('dedup_sha_mismatch') == True, (
    f'Expected dedup_sha_mismatch=True for different-file collision, got {mismatch_data.get("dedup_sha_mismatch")}'
)
print(f'\u2705 dedup_sha_mismatch: {mismatch_data["dedup_sha_mismatch"]} (iOS should log loudly!)')
print(f'   Stored SHA: {CHUNK_1_SHA[:16]}...')
print(f'   Incoming SHA: {different_sha[:16]}...')

In [0]:
print('=' * 60)
print('CHUNKS LIST ENDPOINT')
print('=' * 60)

chunks_resp = session.get(f'/api/captures/{CAPTURE_ID}/audio/chunks')
print(f'HTTP status: {chunks_resp.status_code}')

if chunks_resp.status_code == 200:
    chunks_data = chunks_resp.json()
    print(f'chunk_count: {chunks_data.get("chunk_count")}')
    print(f'is_complete: {chunks_data.get("is_complete")}')
    print()

    chunks = chunks_data.get('chunks', [])
    for chunk in chunks:
        print(f'  chunk_index={chunk["chunk_index"]}, '
              f'upload_id={chunk["upload_id"][:12]}..., '
              f'mime_type={chunk["mime_type"]}, '
              f'size={chunk["size_bytes"]} bytes, '
              f'is_final={chunk["is_final_chunk"]}')

    # Verify ordering
    indices = [c['chunk_index'] for c in chunks]
    assert indices == sorted(indices), f'Chunks not ordered! {indices}'
    print(f'\n\u2705 Chunks returned in order: {indices}')

    # Verify completeness
    assert chunks_data.get('is_complete') == True, 'Expected is_complete=true (chunk 2 has is_final_chunk)'
    print(f'\u2705 is_complete=true (final chunk present)')
elif chunks_resp.status_code == 404:
    print('\u26a0\ufe0f  Endpoint not found \u2014 chunks list route may not be deployed yet')
    print(f'   Response: {chunks_resp.text[:200]}')
else:
    print(f'\u274c Unexpected status: {chunks_resp.status_code}')
    print(f'   Response: {chunks_resp.text[:300]}')

In [0]:
print('=' * 60)
print('AUDIO STREAM CONCAT ENDPOINT')
print('=' * 60)

stream_resp = session.get(f'/api/captures/{CAPTURE_ID}/audio/stream')
print(f'HTTP status: {stream_resp.status_code}')
print(f'Content-Type: {stream_resp.headers.get("Content-Type", "N/A")}')

if stream_resp.status_code == 200:
    body = stream_resp.content
    print(f'Body size: {len(body):,} bytes')

    # For WAV files (single chunk, direct proxy), check RIFF header
    # For M4A concat output, check ftyp box
    if body[:4] == b'RIFF':
        print(f'\u2705 Valid WAV header (single chunk direct proxy)')
    elif body[4:8] == b'ftyp':
        print(f'\u2705 Valid M4A header (ftyp box) \u2014 concat produced valid output')
    elif body[:4] == b'\x00\x00\x00':
        # MP4/M4A often starts with box size bytes
        print(f'First 12 bytes: {body[:12].hex()}')
        if b'ftyp' in body[:32]:
            print(f'\u2705 Valid M4A (ftyp found in first 32 bytes)')
        else:
            print(f'\u26a0\ufe0f  Could not verify format \u2014 first 32 bytes: {body[:32].hex()}')
    else:
        print(f'First 12 bytes: {body[:12].hex()}')
        print(f'\u26a0\ufe0f  Unknown format \u2014 may need ffmpeg for concat verification')

elif stream_resp.status_code == 404:
    print('\u26a0\ufe0f  Endpoint not found \u2014 audio stream route may not be deployed yet')
    print(f'   Response: {stream_resp.text[:200]}')
else:
    print(f'\u274c Unexpected status: {stream_resp.status_code}')
    print(f'   Response: {stream_resp.text[:300]}')

In [0]:
from pyspark.sql import functions as F

otel_table = f'{CATALOG}.{SCHEMA}.lakeloom_ai_otel_logs'
print(f'OTel table: {otel_table}')

try:
    otel_df = spark.table(otel_table)

    chunk_logs = (
        otel_df
        .filter(F.col('time') > F.current_timestamp() - F.expr('INTERVAL 5 MINUTES'))
        .filter(
            (F.col('body').cast('string').contains('chunk_index')) |
            (F.col('body').cast('string').contains('chunk.dedup'))
        )
        .select('time', 'severity_text', F.col('body').cast('string').alias('body_text'))
        .orderBy(F.col('time').desc())
        .limit(10)
    )

    print('Recent chunk-related OTel logs:')
    print('\u2500' * 60)
    display(chunk_logs)
except Exception as e:
    print(f'\u26a0\ufe0f  Could not query OTel logs: {e}')

In [0]:
print('=' * 60)
print('CHUNKED RECORDING TEST SUMMARY')
print('=' * 60)

results = {
    'chunk_0_uploaded': upload_results[0].get('chunk_index') == 0,
    'chunk_1_uploaded': upload_results[1].get('chunk_index') == 1,
    'chunk_2_uploaded': upload_results[2].get('chunk_index') == 2,
    'chunk_2_is_final': upload_results[2].get('is_final_chunk') == True,
    'dedup_same_sha': dedup_data.get('dedup_sha_mismatch') == False,
    'dedup_diff_sha': mismatch_data.get('dedup_sha_mismatch') == True,
    'dedup_returns_201': dedup_resp.status_code == 201,
    'chunks_endpoint_ok': chunks_resp.status_code == 200 if 'chunks_resp' in dir() else False,
    'stream_endpoint_ok': stream_resp.status_code == 200 if 'stream_resp' in dir() else False,
}

all_pass = True
for test_name, passed in results.items():
    status = '\u2705' if passed else '\u274c'
    print(f'  {status} {test_name}')
    if not passed:
        all_pass = False

print()
if all_pass:
    print('ALL TESTS PASSED')
else:
    critical = all([
        results['chunk_0_uploaded'],
        results['chunk_1_uploaded'],
        results['chunk_2_uploaded'],
        results['chunk_2_is_final'],
        results['dedup_returns_201'],
    ])
    if critical:
        print('Core tests passed. Some endpoints not yet deployed (expected pre-merge).')
    else:
        print('CRITICAL TESTS FAILED')
        raise AssertionError('Chunked recording core tests failed')

print(f'\nUpload details:')
print(json.dumps({
    'project_id': PROJECT_ID,
    'capture_id': CAPTURE_ID,
    'chunk_uploads': [{'id': r['id'], 'chunk_index': r.get('chunk_index')} for r in upload_results],
}, indent=2))